# Weekly cross-sectional momentum: replication of Dobrynskaya (2023)

**What this notebook does**
1. Downloads daily prices for 7 cryptocurrencies from Binance (public data, no API key).
2. Converts to weekly (Friday) closes and keeps only the window where **every** coin has data.
3. Each week ranks coins on their trailing average weekly return, goes long the winners and
   short the losers (rank-demeaned, normalized so gross exposure = 1), and holds for one week.
4. Charges **20 bps per unit of one-way turnover** (7 bps commission + 13 bps slippage),
   as the course brief specifies for market orders.
5. Reports returns, volatility, Sharpe, max drawdown, alpha and beta against BTC and an
   equal-weight basket, significance tests, a bootstrap Sharpe interval, cost sensitivity,
   and calendar-year results.

**Choices fixed in advance (not tuned on the results):** weekly data, and a 2-week formation
with a 1-week holding period as the base case (Liu et al. 2022; Yang 2025). Formations of
1-4 weeks are reported as a robustness table and are not used to pick the base case.

Every number in this notebook is printed by code. Nothing is typed in by hand.

In [ ]:
import os
import sys

sys.path.insert(0, os.getcwd())

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from cryptolib import *  # noqa: F401,F403  (load_universe, backtest, metrics, tests, ...)

pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

UNIVERSE = ["BTCUSDT", "ETHUSDT", "ADAUSDT", "BNBUSDT", "XRPUSDT", "DOTUSDT", "MATICUSDT"]
START = "2020-01-01"
END = None                    # None = today
BASE_FORMATION_WEEKS = 2      # fixed a priori
HORIZONS_TESTED = [1, 2, 3, 4]

## 1. Download daily data
Binance returns at most 1,000 candles per request, so `fetch_klines` pages through the history.
Any coin that cannot be downloaded from your location is skipped and reported.

In [ ]:
close_d, qvol_d = load_universe(UNIVERSE, START, END)
print("\nDaily data coverage:")
print(coverage_table(close_d).to_string())

## 2. Weekly prices and the common sample
If a coin has a shorter history (DOT launched in 2020) or its symbol was renamed or delisted
(MATIC migrated to POL in 2024), dropping incomplete rows would silently shorten the sample.
The window is printed explicitly so you can check it before trusting any result.

In [ ]:
close_w_all = to_weekly(close_d)
px = common_window(close_w_all)
ret = px.pct_change().dropna()

print(f"Common weekly window: {px.index[0].date()} -> {px.index[-1].date()}")
print(f"Weekly prices: {len(px)}   weekly returns: {len(ret)}")
print(f"Coins in sample: {list(px.columns)}")
dropped = [c for c in close_w_all.columns if c not in px.columns]
if dropped:
    print(f"Coins with no weekly price in the common window: {dropped}")
last_any = close_w_all.dropna(how="all").index[-1]
if px.index[-1] < last_any - pd.Timedelta(weeks=4):
    print("WARNING: at least one coin ends well before the others, so the sample is truncated.")
    print("         Consider removing that coin from UNIVERSE and re-running.")

## 3. Formation-horizon robustness (1-4 weeks, net of costs)
Dobrynskaya (2023) reports momentum over 1-4 week formations. This table checks that the
result does not depend on a single formation length.

In [ ]:
horizon_bt = {k: momentum_backtest(ret, formation=k) for k in HORIZONS_TESTED}
horizon_table = metrics_table(
    {f"{k}w formation (net)": bt["net"] for k, bt in horizon_bt.items()}, PPY_WEEKLY)
print(horizon_table.to_string())

## 4. Base strategy: gross versus net of costs
`gross` is before costs. `net` deducts 20 bps for every unit of one-way turnover.
Turnover is the sum of absolute weight changes each week.

In [ ]:
base = horizon_bt[BASE_FORMATION_WEEKS]
print(metrics_table({"Gross (no costs)": base["gross"],
                     "Net (20 bps)": base["net"]}, PPY_WEEKLY).to_string())
print()
print(cost_summary(base, PPY_WEEKLY).to_string())
print("\nNet Sharpe at alternative all-in costs (bps per unit turnover):")
sens = pd.Series({f"{b} bps": net_sharpe_at_cost(base, b, PPY_WEEKLY)
                  for b in [0, 10, 20, 30, 40, 60]})
print(sens.to_string())

## 5. Benchmarks over the same window
* **BTC buy-and-hold:** the usual crypto benchmark.
* **Equal-weight basket:** the average weekly return of the sample coins. It is treated as
  costless, so it is slightly optimistic as a benchmark.
The strategy and both benchmarks use identical dates. Earlier comparisons were not aligned.

In [ ]:
btc = ret["BTC"]
basket = ret.mean(axis=1)
print(metrics_table({
    f"Momentum net ({BASE_FORMATION_WEEKS}w)": base["net"],
    "BTC buy & hold": btc,
    "Equal-weight basket": basket,
}, PPY_WEEKLY).to_string())

## 6. Alpha and beta
OLS of the strategy's net return on the benchmark's return, with Newey-West (HAC, 4 lags)
standard errors because weekly returns are autocorrelated. Alpha is annualized as
intercept x 52.

In [ ]:
print("Versus BTC:")
print(alpha_beta(base["net"], btc, PPY_WEEKLY).to_string())
print("\nVersus equal-weight basket:")
print(alpha_beta(base["net"], basket, PPY_WEEKLY).to_string())

## 7. Significance
* Is the net mean weekly return different from zero? (HAC t-test)
* Does the strategy's net mean differ from BTC's? (HAC t-test on the difference)
* 95% moving-block bootstrap interval for the net Sharpe ratio (4-week blocks).

Several horizons and benchmarks are tested, so treat the p-values as descriptive, not as
confirmatory evidence.

In [ ]:
m0 = mean_test_hac(base["net"])
print(f"Net mean weekly return: {m0['mean'] * 100:+.3f}%   t = {m0['t-stat']:.2f}   "
      f"p = {m0['p-value']:.4f}   n = {m0['n']}")
d_btc = mean_diff_test(base["net"], btc)
print(f"Net minus BTC, mean weekly: {d_btc['mean diff'] * 100:+.3f}%   "
      f"t = {d_btc['t-stat']:.2f}   p = {d_btc['p-value']:.4f}   n = {d_btc['n']}")
ci, _ = bootstrap_sharpe(base["net"], PPY_WEEKLY, n_boot=2000, block=4)
base_sharpe = perf_metrics(base["net"], PPY_WEEKLY)["Sharpe"]
print(f"Net Sharpe {base_sharpe:.2f}   95% block-bootstrap CI [{ci[0]:.2f}, {ci[1]:.2f}]")

## 8. Calendar-year robustness
One strong year can carry an entire backtest, so results are split by calendar year.
Years with few weeks (the first and last) are noisy.

In [ ]:
print(yearly_table(base["net"], PPY_WEEKLY).to_string())

## 9. Equity curves and saved output

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
curves = {"Momentum net (20 bps)": base["net"],
          "BTC buy & hold": btc,
          "Equal-weight basket": basket}
for name, r in curves.items():
    (1 + r.fillna(0)).cumprod().plot(ax=ax, label=name)
ax.set_yscale("log")
ax.set_title("Growth of $1 over the common window (log scale)")
ax.set_ylabel("Growth of $1")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("momentum_equity_curves.png", dpi=150)
plt.show()

base.to_csv("momentum_base_backtest.csv")
print("Saved momentum_equity_curves.png and momentum_base_backtest.csv")